# Generative Dog Images — 동결 생성모델 + 인식모델 검증 (Colab)

**연습 기법** (IOAI 2024 *Madarian Cow* 와 동일): **사전학습된 생성모델을 그대로(동결)** 써서 이미지를 만들고,
**별도의 인식모델로 결과를 검증·채점**한다. (Madarian Cow: miniSD 디퓨전으로 생성 → DETR 로 객체검출해 채점)

**시나리오 매핑**:
- Madarian Cow: 동결 디퓨전(miniSD)으로 생성 → **DETR** 가 올바른 객체(소화전 포함/제외)를 검출하면 정답.
- 여기(Generative Dog Images): 동결 디퓨전(Stable Diffusion)으로 **개 이미지 생성** → **ImageNet 분류기**가
  '개'로 인식하면 정답. 캐글은 생성 이미지를 인식모델(InceptionV3, MiFID)로 채점 — 같은 *생성→인식* 패턴.

**코드는 기본만**: `diffusers`(동결 생성) + `torchvision`(동결 분류기). **학습 없음** — 사전학습 모델만 사용.


## 0. 라이브러리 설치 & Kaggle 자격증명(제출용)


In [ ]:
import sys, subprocess
for pkg in ["diffusers", "accelerate", "torch", "torchvision", "transformers", "kaggle"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")


In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")


## 1. 동결 생성모델 (Stable Diffusion) — *학습하지 않음*
Madarian Cow 의 miniSD 처럼, 사전학습 디퓨전을 그대로 불러와 개 이미지를 생성한다.
`sd-turbo` 는 1~2스텝으로 빠르게 생성 → Colab 에서 가볍게 연습 가능.


In [ ]:
import torch, numpy as np
from diffusers import AutoPipelineForText2Image

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)
pipe.set_progress_bar_config(disable=True)
print("동결 생성모델 로드 완료 (sd-turbo)")


## 2. 동결 인식모델 (ImageNet 분류기) — Madarian Cow 의 DETR 역할
생성 이미지를 분류기에 통과시켜 **'개'로 인식되는지** 판정(`is_dog`). ImageNet 의 개 품종 클래스는 151~268.


In [ ]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from PIL import Image

_w = MobileNet_V2_Weights.IMAGENET1K_V1
clf = mobilenet_v2(weights=_w).to(device).eval()
_tf = _w.transforms()
DOG_CLASSES = set(range(151, 269))          # ImageNet 개 품종 인덱스

def is_dog(img):
    x = _tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = clf(x).softmax(1)[0]
    return int(pred.argmax()) in DOG_CLASSES   # 최상위 예측이 개 품종이면 True

print("동결 인식모델 로드 완료 (mobilenet_v2, ImageNet)")


## 3. 생성 → 검증 루프 (Madarian Cow 패턴)
여러 프롬프트로 개 이미지를 생성하고, 인식모델이 '개'로 인정한 비율을 점수로 본다.
(데모용으로 N=64 생성. 실제 제출은 더 많이 생성할수록 좋다.)


In [ ]:
import random
breeds = ["golden retriever","german shepherd","beagle","bulldog","poodle","husky",
          "labrador","corgi","dalmatian","chihuahua","boxer","pug"]
N = 64
images, recognized = [], 0
for i in range(N):
    prompt = f"a photo of a {random.choice(breeds)} dog, centered, full body"
    img = pipe(prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
    images.append(img)
    if is_dog(img):
        recognized += 1
print(f"생성 {N}장 중 인식모델이 '개'로 인정: {recognized}/{N} = {recognized/N:.3f}")
print("→ Madarian Cow 처럼 '생성물이 인식모델에 올바르게 인식되는가' 가 점수 (높을수록 좋음).")


## 4. 캐글 제출 파일 생성 (`images.zip`, 64×64)
Generative Dog Images 는 생성 이미지 묶음(`images.zip`)을 제출하면 InceptionV3 기반 MiFID 로 채점한다.


In [ ]:
import os, zipfile, shutil
out_dir = "images"
if os.path.isdir(out_dir): shutil.rmtree(out_dir)
os.makedirs(out_dir)
for i, img in enumerate(images):
    img.resize((64, 64)).save(os.path.join(out_dir, f"dog_{i:05d}.png"))
with zipfile.ZipFile("images.zip", "w") as zf:
    for f in sorted(os.listdir(out_dir)):
        zf.write(os.path.join(out_dir, f), f)
print("images.zip 생성:", len(images), "장 (64x64)")


In [ ]:
try:
    from google.colab import files
    files.download("images.zip")
except Exception as e:
    print("자동 다운로드 불가:", e, "| images.zip 위치 확인")


## 🏁 제출하기
`images.zip` 을 [Generative Dog Images](https://www.kaggle.com/c/generative-dog-images) 에 제출하세요(생성 이미지가 많을수록 MiFID 점수에 유리).

**기법 요약**: 생성모델을 **학습하지 않고(동결)** 그대로 써서 이미지를 만들고, **별도 인식모델로 검증/채점** — IOAI *Madarian Cow* 와 동일한 *생성→인식* 패턴.
더 발전시키려면: 프롬프트 엔지니어링, 더 좋은 디퓨전/스텝수, 인식모델이 떨어뜨린 이미지 필터링(Madarian Cow 의 'magic' 처럼 생성 조건을 조정).

> 참고: Generative Dog Images 는 종료된 대회라 실시간 리더보드 점수는 제한될 수 있습니다. 같은 *생성→인식* 기법을 연습하는 데 목적이 있으며, 제출 파일 형식(`images.zip`)은 그대로 유효합니다.
